# 08 - Profiling e Qualidade da Camada Prata

## Objetivo

Avaliar a qualidade dos dados após as transformações realizadas na
camada Prata.

Serão analisadas as seguintes tabelas:

- `mvp3_eng_dados.prata.imdb_avaliacoes`
- `mvp3_eng_dados.prata.rt_filmes`
- `mvp3_eng_dados.prata.rt_avaliacoes`
- `mvp3_eng_dados.prata.avaliacoes_harmonizadas`

## Dimensões de qualidade avaliadas

- Completude
- Consistência
- Unicidade
- Integridade referencial
- Validade dos domínios
- Distribuição dos dados
- Possíveis valores extremos

Este notebook possui finalidade exclusivamente analítica.

Nenhuma transformação ou correção dos dados será realizada nesta etapa.

In [0]:
from pyspark.sql import functions as F

TABELA_IMDB = "mvp3_eng_dados.prata.imdb_avaliacoes"
TABELA_RT_FILMES = "mvp3_eng_dados.prata.rt_filmes"
TABELA_RT_AVALIACOES = "mvp3_eng_dados.prata.rt_avaliacoes"
TABELA_HARMONIZADA = "mvp3_eng_dados.prata.avaliacoes_harmonizadas"

print("Configuração concluída.")

Configuração concluída.


In [0]:
df_imdb = spark.table(TABELA_IMDB)
df_rt_filmes = spark.table(TABELA_RT_FILMES)
df_rt_avaliacoes = spark.table(TABELA_RT_AVALIACOES)
df_harmonizada = spark.table(TABELA_HARMONIZADA)

print("Tabelas Prata carregadas.")

Tabelas Prata carregadas.


In [0]:
qtd_imdb = df_imdb.count()
qtd_rt_filmes = df_rt_filmes.count()
qtd_rt_avaliacoes = df_rt_avaliacoes.count()
qtd_harmonizada = df_harmonizada.count()

print(f"IMDb avaliações:          {qtd_imdb:,}")
print(f"RT filmes:                {qtd_rt_filmes:,}")
print(f"RT avaliações:            {qtd_rt_avaliacoes:,}")
print(f"Avaliações harmonizadas:  {qtd_harmonizada:,}")

IMDb avaliações:          49,582
RT filmes:                17,712
RT avaliações:            1,010,546
Avaliações harmonizadas:  1,060,128


In [0]:
def analisar_nulos(df):
    total = df.count()

    expressoes = []

    for coluna in df.columns:
        expressoes.append(
            F.sum(
                F.when(
                    F.col(coluna).isNull(),
                    1
                ).otherwise(0)
            ).alias(coluna)
        )

    resultado = df.agg(*expressoes)

    return resultado, total

In [0]:
nulos_imdb, total_imdb = analisar_nulos(df_imdb)

print(f"Total de registros IMDb: {total_imdb:,}")

display(nulos_imdb)

Total de registros IMDb: 49,582


id_avaliacao,review,sentiment,sentimento,fonte_dados,data_hora_processamento
0,0,0,0,0,0


In [0]:
nulos_rt_filmes, total_rt_filmes = analisar_nulos(df_rt_filmes)

print(
    f"Total de registros RT Filmes: "
    f"{total_rt_filmes:,}"
)

display(nulos_rt_filmes)

Total de registros RT Filmes: 17,712


id_filme,rotten_tomatoes_link,movie_title,data_lancamento_original,data_lancamento_streaming,duracao_minutos,avaliacao_tomatometer,quantidade_tomatometer,avaliacao_audiencia,quantidade_audiencia,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,streaming_release_date,runtime,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count,fonte_dados,data_hora_processamento
0,0,0,1166,384,314,44,44,296,297,321,8578,0,19,194,1542,352,1166,384,314,499,44,44,44,448,296,297,0,0,0,0,0


In [0]:
nulos_rt_avaliacoes, total_rt_avaliacoes = analisar_nulos(
    df_rt_avaliacoes
)

print(
    f"Total de registros RT Avaliações: "
    f"{total_rt_avaliacoes:,}"
)

display(nulos_rt_avaliacoes)

Total de registros RT Avaliações: 1,010,546


id_avaliacao,rotten_tomatoes_link,critic_name,top_critic,eh_top_critic,publisher_name,review_type,sentimento,review_score,review_date,data_avaliacao,review_content,fonte_dados,data_hora_processamento
0,0,16454,0,0,0,0,0,273605,0,0,58526,0,0


In [0]:
nulos_harmonizada, total_harmonizada = analisar_nulos(
    df_harmonizada
)

print(
    f"Total de registros harmonizados: "
    f"{total_harmonizada:,}"
)

display(nulos_harmonizada)

Total de registros harmonizados: 1,060,128


id_avaliacao_global,id_avaliacao,fonte,tipo_avaliador,id_filme,titulo_filme,link_filme,texto_avaliacao,classificacao_original,sentimento,nota_original,data_avaliacao,nome_avaliador,publicacao,eh_top_critic,data_hora_integracao
0,0,0,0,49712,49712,49582,58526,0,0,323187,49582,66036,49582,49582,0


In [0]:
id_imdb_total = df_imdb.count()
id_imdb_distintos = (
    df_imdb
    .select("id_avaliacao")
    .distinct()
    .count()
)

id_rt_total = df_rt_avaliacoes.count()
id_rt_distintos = (
    df_rt_avaliacoes
    .select("id_avaliacao")
    .distinct()
    .count()
)

id_global_total = df_harmonizada.count()
id_global_distintos = (
    df_harmonizada
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

print("IMDb")
print(f"Registros:       {id_imdb_total:,}")
print(f"IDs distintos:   {id_imdb_distintos:,}")

print("\nRotten Tomatoes")
print(f"Registros:       {id_rt_total:,}")
print(f"IDs distintos:   {id_rt_distintos:,}")

print("\nTabela harmonizada")
print(f"Registros:       {id_global_total:,}")
print(f"IDs distintos:   {id_global_distintos:,}")

IMDb
Registros:       49,582
IDs distintos:   49,582

Rotten Tomatoes
Registros:       1,010,546
IDs distintos:   1,010,546

Tabela harmonizada
Registros:       1,060,128
IDs distintos:   1,060,128


In [0]:
assert id_imdb_total == id_imdb_distintos, (
    "Existem IDs duplicados na tabela IMDb."
)

assert id_rt_total == id_rt_distintos, (
    "Existem IDs duplicados na tabela RT Avaliações."
)

assert id_global_total == id_global_distintos, (
    "Existem IDs globais duplicados."
)

print("✓ Identificadores únicos validados.")

✓ Identificadores únicos validados.


In [0]:
display(
    df_harmonizada
    .groupBy(
        "fonte",
        "sentimento"
    )
    .count()
    .orderBy(
        "fonte",
        "sentimento"
    )
)

fonte,sentimento,count
IMDb,NEGATIVO,24698
IMDb,POSITIVO,24884
Rotten Tomatoes,NEGATIVO,366614
Rotten Tomatoes,POSITIVO,643932


In [0]:
sentimentos_invalidos = (
    df_harmonizada
    .filter(
        ~F.col("sentimento").isin(
            "POSITIVO",
            "NEGATIVO"
        )
        |
        F.col("sentimento").isNull()
    )
    .count()
)

print(
    f"Sentimentos inválidos: "
    f"{sentimentos_invalidos:,}"
)

assert sentimentos_invalidos == 0, (
    "Foram encontrados sentimentos fora do domínio esperado."
)

print("✓ Domínio de sentimento validado.")

Sentimentos inválidos: 0
✓ Domínio de sentimento validado.


In [0]:
distribuicao_fontes = (
    df_harmonizada
    .groupBy("fonte")
    .count()
    .orderBy("fonte")
)

display(distribuicao_fontes)

fonte,count
IMDb,49582
Rotten Tomatoes,1010546


In [0]:
links_sem_filme = (
    df_rt_avaliacoes
    .select("rotten_tomatoes_link")
    .distinct()
    .join(
        df_rt_filmes
        .select("rotten_tomatoes_link")
        .distinct(),
        on="rotten_tomatoes_link",
        how="left_anti"
    )
)

qtd_links_sem_filme = links_sem_filme.count()

print(
    f"Links de avaliações sem filme correspondente: "
    f"{qtd_links_sem_filme:,}"
)

Links de avaliações sem filme correspondente: 6


In [0]:
display(
    links_sem_filme
    .orderBy("rotten_tomatoes_link")
)

rotten_tomatoes_link
m/+_one_2019
m/+h
m/-_man
m/-cule_valley_of_the_lost_ants
m/patton_oswalt_tragedy_+_comedy_equals_time
m/sympathy-for-the-devil-one-+-one


In [0]:
estatisticas_texto = (
    df_harmonizada
    .withColumn(
        "tamanho_texto",
        F.length(
            F.col("texto_avaliacao")
        )
    )
    .groupBy("fonte")
    .agg(
        F.count("*").alias("registros"),
        F.min("tamanho_texto").alias("tamanho_minimo"),
        F.avg("tamanho_texto").alias("tamanho_medio"),
        F.expr(
            "percentile_approx(tamanho_texto, 0.5)"
        ).alias("mediana"),
        F.expr(
            "percentile_approx(tamanho_texto, 0.95)"
        ).alias("percentil_95"),
        F.max("tamanho_texto").alias("tamanho_maximo")
    )
    .orderBy("fonte")
)

display(estatisticas_texto)

fonte,registros,tamanho_minimo,tamanho_medio,mediana,percentil_95,tamanho_maximo
IMDb,49582,32,1310.5682304061959,971,3394,13704
Rotten Tomatoes,1010546,1,128.12389655679502,127,226,258


In [0]:
textos_vazios = (
    df_harmonizada
    .filter(
        F.col("texto_avaliacao").isNull()
        |
        (F.length(F.trim(F.col("texto_avaliacao"))) == 0)
    )
    .count()
)

print(
    f"Avaliações com texto nulo ou vazio: "
    f"{textos_vazios:,}"
)

Avaliações com texto nulo ou vazio: 58,526


In [0]:
df_datas_rt = (
    df_rt_avaliacoes
    .select("data_avaliacao")
)

total_datas_rt = df_datas_rt.count()

datas_nulas = (
    df_datas_rt
    .filter(
        F.col("data_avaliacao").isNull()
    )
    .count()
)

datas_validas = total_datas_rt - datas_nulas

print(f"Registros RT:       {total_datas_rt:,}")
print(f"Datas válidas:      {datas_validas:,}")
print(f"Datas nulas:        {datas_nulas:,}")

Registros RT:       1,010,546
Datas válidas:      1,010,546
Datas nulas:        0


In [0]:

# ============================================================
# Qualidade da conversão das notas do Rotten Tomatoes
# ============================================================

print("=" * 70)
print("QUALIDADE DA CONVERSÃO DAS NOTAS - ROTTEN TOMATOES")
print("=" * 70)

print("\nColunas disponíveis em rt_avaliacoes:")
print(df_rt_avaliacoes.columns)

####-----

QUALIDADE DA CONVERSÃO DAS NOTAS - ROTTEN TOMATOES

Colunas disponíveis em rt_avaliacoes:
['id_avaliacao', 'rotten_tomatoes_link', 'critic_name', 'top_critic', 'eh_top_critic', 'publisher_name', 'review_type', 'sentimento', 'review_score', 'review_date', 'data_avaliacao', 'review_content', 'fonte_dados', 'data_hora_processamento']


In [0]:
# ============================================================
# Identificação dos campos relacionados à nota
# ============================================================

colunas_rt = set(df_rt_avaliacoes.columns)

if "review_score" in colunas_rt:
    coluna_nota_original = "review_score"
elif "nota_original" in colunas_rt:
    coluna_nota_original = "nota_original"
else:
    coluna_nota_original = None

if "nota_numerica" in colunas_rt:
    coluna_nota_convertida = "nota_numerica"
elif "nota_critico" in colunas_rt:
    coluna_nota_convertida = "nota_critico"
else:
    coluna_nota_convertida = None

print(
    "Campo da nota original:   ",
    coluna_nota_original
)

print(
    "Campo da nota convertida:  ",
    coluna_nota_convertida
)

Campo da nota original:    review_score
Campo da nota convertida:   None


In [0]:
# ============================================================
# Qualidade do campo review_score
# ============================================================

from pyspark.sql import functions as F

total_rt = df_rt_avaliacoes.count()

scores_preenchidos = (
    df_rt_avaliacoes
    .filter(
        F.col("review_score").isNotNull()
        &
        (F.trim(F.col("review_score")) != "")
    )
    .count()
)

scores_ausentes = total_rt - scores_preenchidos

percentual_preenchido = (
    scores_preenchidos / total_rt * 100
    if total_rt > 0
    else 0
)

print(f"Registros RT:             {total_rt:,}")
print(f"review_score preenchidos: {scores_preenchidos:,}")
print(f"review_score ausentes:    {scores_ausentes:,}")
print(f"Percentual preenchido:    {percentual_preenchido:.2f}%")

Registros RT:             1,010,546
review_score preenchidos: 736,941
review_score ausentes:    273,605
Percentual preenchido:    72.93%


In [0]:
# ============================================================
# Valores mais frequentes de review_score
# ============================================================

display(
    df_rt_avaliacoes
    .filter(
        F.col("review_score").isNotNull()
        &
        (F.trim(F.col("review_score")) != "")
    )
    .groupBy("review_score")
    .count()
    .orderBy(
        F.desc("count"),
        F.asc("review_score")
    )
    .limit(30)
)

review_score,count
3/5,80899
4/5,74944
3/4,64529
2/5,53839
2/4,42440
2.5/4,40119
3.5/4,30603
3.5/5,27663
B,21691
5/5,21631


In [0]:
# ============================================================
# Classificação dos formatos de review_score
# ============================================================

df_formatos_score = (
    df_rt_avaliacoes
    .filter(
        F.col("review_score").isNotNull()
        &
        (F.trim(F.col("review_score")) != "")
    )
    .withColumn(
        "formato_score",
        F.when(
            F.col("review_score").rlike(r"^\s*\d+(\.\d+)?\s*/\s*\d+(\.\d+)?\s*$"),
            F.lit("FRACAO")
        )
        .when(
            F.col("review_score").rlike(r"^\s*\d+(\.\d+)?\s*%\s*$"),
            F.lit("PERCENTUAL")
        )
        .when(
            F.col("review_score").rlike(r"^\s*[A-Fa-f][+-]?\s*$"),
            F.lit("LETRA")
        )
        .when(
            F.col("review_score").rlike(r"^\s*\d+(\.\d+)?\s*$"),
            F.lit("NUMERICO")
        )
        .otherwise(
            F.lit("OUTRO")
        )
    )
)

display(
    df_formatos_score
    .groupBy("formato_score")
    .count()
    .orderBy(F.desc("count"))
)

formato_score,count
FRACAO,620121
LETRA,115397
NUMERICO,1420
OUTRO,3


In [0]:
# ============================================================
# Exemplos dos diferentes formatos de score
# ============================================================

display(
    df_formatos_score
    .select(
        "review_score",
        "formato_score"
    )
    .distinct()
    .orderBy(
        "formato_score",
        "review_score"
    )
    .limit(100)
)

review_score,formato_score
0.02/5,FRACAO
0.05/4,FRACAO
0.1/5,FRACAO
0.13/1,FRACAO
0.13/5,FRACAO
0.2/5,FRACAO
0.21/10,FRACAO
0.25/4,FRACAO
0.3/5,FRACAO
0.4/5,FRACAO


In [0]:
# ============================================================
# Distribuição de sentimento por fonte
# ============================================================

display(
    df_harmonizada
    .groupBy(
        "fonte",
        "sentimento"
    )
    .count()
    .orderBy(
        "fonte",
        "sentimento"
    )
)

fonte,sentimento,count
IMDb,NEGATIVO,24698
IMDb,POSITIVO,24884
Rotten Tomatoes,NEGATIVO,366614
Rotten Tomatoes,POSITIVO,643932


In [0]:
# ============================================================
# Distribuição percentual de sentimento por fonte
# ============================================================

from pyspark.sql.window import Window

janela_fonte = Window.partitionBy("fonte")

df_distribuicao_sentimento = (
    df_harmonizada
    .groupBy(
        "fonte",
        "sentimento"
    )
    .count()
    .withColumn(
        "total_fonte",
        F.sum("count").over(janela_fonte)
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("count") / F.col("total_fonte") * 100,
            2
        )
    )
)

display(
    df_distribuicao_sentimento
    .orderBy(
        "fonte",
        "sentimento"
    )
)

fonte,sentimento,count,total_fonte,percentual
IMDb,NEGATIVO,24698,49582,49.81
IMDb,POSITIVO,24884,49582,50.19
Rotten Tomatoes,NEGATIVO,366614,1010546,36.28
Rotten Tomatoes,POSITIVO,643932,1010546,63.72


In [0]:
# ============================================================
# Consistência do domínio de sentimento
# ============================================================

sentimentos_validos = [
    "POSITIVO",
    "NEGATIVO"
]

sentimentos_invalidos = (
    df_harmonizada
    .filter(
        F.col("sentimento").isNull()
        |
        (~F.col("sentimento").isin(sentimentos_validos))
    )
    .count()
)

print(
    f"Sentimentos fora do domínio esperado: "
    f"{sentimentos_invalidos:,}"
)

assert sentimentos_invalidos == 0, (
    "Existem sentimentos fora do domínio esperado."
)

print("✓ Domínio de sentimento validado.")

Sentimentos fora do domínio esperado: 0
✓ Domínio de sentimento validado.


In [0]:
# ============================================================
# Completude dos campos essenciais
# ============================================================

campos_essenciais = [
    "id_avaliacao_global",
    "fonte",
    "texto_avaliacao",
    "sentimento"
]

resultados_completude = []

total_harmonizado = df_harmonizada.count()

for campo in campos_essenciais:

    ausentes = (
        df_harmonizada
        .filter(
            F.col(campo).isNull()
            |
            (F.trim(F.col(campo).cast("string")) == "")
        )
        .count()
    )

    percentual_completo = (
        ((total_harmonizado - ausentes) / total_harmonizado) * 100
        if total_harmonizado > 0
        else 0
    )

    resultados_completude.append(
        (
            campo,
            ausentes,
            round(percentual_completo, 2)
        )
    )

df_completude = spark.createDataFrame(
    resultados_completude,
    [
        "campo",
        "registros_ausentes",
        "percentual_completo"
    ]
)

display(df_completude)

campo,registros_ausentes,percentual_completo
id_avaliacao_global,0,100.0
fonte,0,100.0
texto_avaliacao,58526,94.48
sentimento,0,100.0


In [0]:
# ============================================================
# Unicidade do identificador global
# ============================================================

total_registros = df_harmonizada.count()

ids_globais_distintos = (
    df_harmonizada
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

duplicidades_id_global = (
    total_registros - ids_globais_distintos
)

print(f"Registros:                  {total_registros:,}")
print(f"IDs globais distintos:      {ids_globais_distintos:,}")
print(f"Duplicidades de ID global:  {duplicidades_id_global:,}")

assert duplicidades_id_global == 0, (
    "Foram encontrados IDs globais duplicados."
)

print("✓ Unicidade do identificador global validada.")

Registros:                  1,060,128
IDs globais distintos:      1,060,128
Duplicidades de ID global:  0
✓ Unicidade do identificador global validada.


In [0]:
# ============================================================
# Integridade referencial RT avaliações × RT filmes
# ============================================================

df_rt_links_avaliacoes = (
    df_rt_avaliacoes
    .select("rotten_tomatoes_link")
    .filter(
        F.col("rotten_tomatoes_link").isNotNull()
    )
    .distinct()
)

df_rt_links_filmes = (
    df_rt_filmes
    .select("rotten_tomatoes_link")
    .filter(
        F.col("rotten_tomatoes_link").isNotNull()
    )
    .distinct()
)

links_sem_filme = (
    df_rt_links_avaliacoes
    .join(
        df_rt_links_filmes,
        on="rotten_tomatoes_link",
        how="left_anti"
    )
)

qtd_links_sem_filme = links_sem_filme.count()

print(
    f"Links distintos de avaliações sem filme correspondente: "
    f"{qtd_links_sem_filme:,}"
)

Links distintos de avaliações sem filme correspondente: 6


In [0]:
# ============================================================
# Inspeção dos links sem filme correspondente
# ============================================================

display(
    links_sem_filme
    .orderBy("rotten_tomatoes_link")
)

rotten_tomatoes_link
m/+_one_2019
m/+h
m/-_man
m/-cule_valley_of_the_lost_ants
m/patton_oswalt_tragedy_+_comedy_equals_time
m/sympathy-for-the-devil-one-+-one


In [0]:
# ============================================================
# Avaliações associadas aos links sem filme
# ============================================================

avaliacoes_sem_filme = (
    df_rt_avaliacoes
    .join(
        links_sem_filme,
        on="rotten_tomatoes_link",
        how="inner"
    )
    .count()
)

percentual_avaliacoes_sem_filme = (
    avaliacoes_sem_filme / df_rt_avaliacoes.count() * 100
    if df_rt_avaliacoes.count() > 0
    else 0
)

print(
    f"Avaliações associadas a links sem filme: "
    f"{avaliacoes_sem_filme:,}"
)

print(
    f"Percentual das avaliações RT: "
    f"{percentual_avaliacoes_sem_filme:.6f}%"
)

Avaliações associadas a links sem filme: 130
Percentual das avaliações RT: 0.012864%


In [0]:
# ============================================================
# Estatísticas de comprimento do texto
# ============================================================

df_tamanho_texto = (
    df_harmonizada
    .withColumn(
        "tamanho_texto",
        F.length(
            F.col("texto_avaliacao")
        )
    )
)

display(
    df_tamanho_texto
    .groupBy("fonte")
    .agg(
        F.count("*").alias("registros"),
        F.round(
            F.avg("tamanho_texto"),
            2
        ).alias("media_caracteres"),
        F.min("tamanho_texto").alias("min_caracteres"),
        F.expr(
            "percentile_approx(tamanho_texto, 0.5)"
        ).alias("mediana_caracteres"),
        F.expr(
            "percentile_approx(tamanho_texto, 0.95)"
        ).alias("percentil_95"),
        F.max("tamanho_texto").alias("max_caracteres")
    )
    .orderBy("fonte")
)

fonte,registros,media_caracteres,min_caracteres,mediana_caracteres,percentil_95,max_caracteres
IMDb,49582,1310.57,32,971,3394,13704
Rotten Tomatoes,1010546,128.12,1,127,226,258


In [0]:
# ============================================================
# Identificação de avaliações muito curtas
# ============================================================

limite_texto_curto = 10

df_textos_curtos = (
    df_harmonizada
    .filter(
        F.length(
            F.trim(
                F.col("texto_avaliacao")
            )
        ) < limite_texto_curto
    )
)

qtd_textos_curtos = df_textos_curtos.count()

print(
    f"Avaliações com menos de {limite_texto_curto} caracteres: "
    f"{qtd_textos_curtos:,}"
)

display(
    df_textos_curtos
    .select(
        "fonte",
        "texto_avaliacao",
        "sentimento"
    )
    .limit(20)
)

Avaliações com menos de 10 caracteres: 850


fonte,texto_avaliacao,sentimento
Rotten Tomatoes,Dire.,NEGATIVO
Rotten Tomatoes,Yawn.,NEGATIVO
Rotten Tomatoes,Godawful.,NEGATIVO
Rotten Tomatoes,asinine,NEGATIVO
Rotten Tomatoes,.,NEGATIVO
Rotten Tomatoes,Muddled.,NEGATIVO
Rotten Tomatoes,Waaaah!,NEGATIVO
Rotten Tomatoes,Odious.,NEGATIVO
Rotten Tomatoes,N.O.,NEGATIVO
Rotten Tomatoes,Boreshot,NEGATIVO


In [0]:
# ============================================================
# Resumo das principais métricas de qualidade
# ============================================================

print("=" * 65)
print("PROFILING DE QUALIDADE — CAMADA PRATA")
print("=" * 65)

print(
    f"Registros harmonizados:                 "
    f"{total_registros:,}"
)

print(
    f"IDs globais distintos:                 "
    f"{ids_globais_distintos:,}"
)

print(
    f"Duplicidades de ID global:              "
    f"{duplicidades_id_global:,}"
)

print(
    f"Sentimentos fora do domínio:            "
    f"{sentimentos_invalidos:,}"
)

print(
    f"Links RT sem filme correspondente:      "
    f"{qtd_links_sem_filme:,}"
)

print(
    f"Avaliações RT associadas a esses links: "
    f"{avaliacoes_sem_filme:,}"
)

print(
    f"Reviews RT com score preenchido:        "
    f"{scores_preenchidos:,}"
)

print(
    f"Reviews RT sem score:                   "
    f"{scores_ausentes:,}"
)

print(
    f"Textos com menos de "
    f"{limite_texto_curto} caracteres:         "
    f"{qtd_textos_curtos:,}"
)

print("=" * 65)

PROFILING DE QUALIDADE — CAMADA PRATA
Registros harmonizados:                 1,060,128
IDs globais distintos:                 1,060,128
Duplicidades de ID global:              0
Sentimentos fora do domínio:            0
Links RT sem filme correspondente:      6
Avaliações RT associadas a esses links: 130
Reviews RT com score preenchido:        736,941
Reviews RT sem score:                   273,605
Textos com menos de 10 caracteres:         850


In [0]:
# ============================================================
# Validações críticas da camada Prata
# ============================================================

assert total_registros > 0, (
    "A tabela harmonizada está vazia."
)

assert ids_globais_distintos == total_registros, (
    "O identificador global não é único."
)

assert sentimentos_invalidos == 0, (
    "Existem sentimentos fora do domínio esperado."
)

print("✓ tabela harmonizada possui registros")
print("✓ identificador global é único")
print("✓ domínio de sentimento está consistente")

if qtd_links_sem_filme > 0:
    print(
        f"⚠ foram identificados "
        f"{qtd_links_sem_filme:,} links RT "
        f"sem filme correspondente"
    )

if scores_ausentes > 0:
    print(
        f"⚠ foram identificadas "
        f"{scores_ausentes:,} avaliações RT "
        f"sem review_score"
    )

print("\n✓ Validações críticas concluídas.")

✓ tabela harmonizada possui registros
✓ identificador global é único
✓ domínio de sentimento está consistente
⚠ foram identificados 6 links RT sem filme correspondente
⚠ foram identificadas 273,605 avaliações RT sem review_score

✓ Validações críticas concluídas.


## Interpretação do profiling da camada Prata

O profiling da camada Prata avaliou as dimensões de completude,
consistência, unicidade e integridade dos dados após os processos
de limpeza, padronização e harmonização.

### Principais verificações

- unicidade do identificador global das avaliações;
- completude dos campos essenciais;
- consistência do domínio de sentimento;
- distribuição das classificações por fonte;
- integridade referencial entre avaliações e filmes do Rotten Tomatoes;
- disponibilidade e heterogeneidade do campo `review_score`;
- identificação de possíveis outliers relacionados ao comprimento
  dos textos das avaliações.

O campo `review_score` foi mantido em seu formato original, pois a
fonte apresenta diferentes escalas e representações de notas.
Uma eventual normalização numérica exigiria regras adicionais de
negócio e não foi realizada durante o profiling.

Os registros sem correspondência entre avaliações e filmes do
Rotten Tomatoes foram mantidos e documentados como achados de
qualidade, evitando a exclusão automática de informação potencialmente
útil.

Os resultados obtidos neste notebook servirão como evidência da
qualidade da camada Prata e como subsídio para a construção da
camada Ouro.

In [0]:
# ============================================================
# Encerramento do notebook
# ============================================================

print("=" * 65)
print("08_PROFILING_QUALIDADE_PRATA — CONCLUÍDO")
print("=" * 65)

print(
    "\nDimensões avaliadas:"
)

print("✓ completude")
print("✓ consistência")
print("✓ unicidade")
print("✓ integridade referencial")
print("✓ possíveis outliers")

print(
    "\nPrincipais conjuntos avaliados:"
)

print("✓ avaliações IMDb")
print("✓ avaliações Rotten Tomatoes")
print("✓ filmes Rotten Tomatoes")
print("✓ avaliações harmonizadas")

print(
    "\nA camada Prata está preparada "
    "para a etapa de modelagem da camada Ouro."
)

08_PROFILING_QUALIDADE_PRATA — CONCLUÍDO

Dimensões avaliadas:
✓ completude
✓ consistência
✓ unicidade
✓ integridade referencial
✓ possíveis outliers

Principais conjuntos avaliados:
✓ avaliações IMDb
✓ avaliações Rotten Tomatoes
✓ filmes Rotten Tomatoes
✓ avaliações harmonizadas

A camada Prata está preparada para a etapa de modelagem da camada Ouro.
